# \# 01 — Model Catalog Ingestion

Fetches model catalog metadata from any OpenAI\-compatible /v1/models endpoint\. Supports hybrid execution \(headless cron jobs via environment variables or interactive UI widgets\) with strict fail\-fast validation and automated snapshot vaulting into datasets/\.

## \#\# Ingestion Parameters

In [1]:
provider_name = 'nvidia'

In [2]:
api_endpoint = 'https://integrate.api.nvidia.com/v1'

In [3]:
api_key = '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'

In [4]:
save_model_data = True

## \#\# Fetch & Validate Catalog

In [5]:
import os
import json
import requests
from datetime import datetime
from pathlib import Path

# 1. Parameter Resolution (Environment variables take precedence for headless cron, UI widgets fall back)
env_provider = os.getenv("PROVIDER_NAME", "").strip()
env_endpoint = os.getenv("API_ENDPOINT", "").strip()
env_key = os.getenv("API_KEY", "").strip()
env_save = os.getenv("SAVE_MODEL_DATA")

resolved_provider = env_provider or (provider_name.strip() if 'provider_name' in globals() and provider_name else "")
resolved_endpoint = env_endpoint or (api_endpoint.strip() if 'api_endpoint' in globals() and api_endpoint else "")
resolved_key = env_key or (api_key.strip() if 'api_key' in globals() and api_key else "")
resolved_save = (env_save.lower() in ("true", "1", "yes")) if env_save is not None else (save_model_data if 'save_model_data' in globals() else True)

# 2. Fail Fast: Validate Required Parameters
missing = []
if not resolved_provider:
    missing.append("provider_name")
if not resolved_endpoint:
    missing.append("api_endpoint")
if not resolved_key:
    missing.append("api_key (or API_KEY env var)")

if missing:
    diag = (
        f"\n--- [DIAGNOSTIC STATE] ---\n"
        f"  globals().get('provider_name') = {repr(globals().get('provider_name'))}\n"
        f"  globals().get('api_endpoint')  = {repr(globals().get('api_endpoint'))}\n"
        f"  globals().get('api_key')       = {repr('***' if globals().get('api_key') else '')}\n"
        f"  os.environ.get('PROVIDER_NAME') = {repr(os.getenv('PROVIDER_NAME'))}\n"
        f"  os.environ.get('API_ENDPOINT')  = {repr(os.getenv('API_ENDPOINT'))}\n"
        f"  os.environ.get('API_KEY')       = {repr('***' if os.getenv('API_KEY') else '')}\n"
        f"---------------------------\n"
        f"Note: If you just typed into an input widget, ensure that the input widget block has executed "
        f"(or click 'Run all' / Shift+Enter on the input cells) so the Python kernel registers the values."
    )
    raise ValueError(f"[FAIL FAST] Missing required configuration parameters: {', '.join(missing)}{diag}")

# 3. HTTP Request & Envelope Validation
url = f"{resolved_endpoint.rstrip('/')}/models"
headers = {"Authorization": f"Bearer {resolved_key}"}

print(f"[Ingestion] Fetching models for '{resolved_provider}' from: {url}")
res = requests.get(url, headers=headers, timeout=25)
res.raise_for_status()

try:
    payload = res.json()
except Exception as e:
    raise RuntimeError(f"[FAIL FAST] Response from {url} is not valid JSON: {res.text[:200]}") from e

# Validate /v1/models structure
if isinstance(payload, dict):
    models = payload.get("data")
    if models is None or not isinstance(models, list):
        raise RuntimeError(f"[FAIL FAST] Unexpected JSON structure from {url}. Expected 'data' key containing a list, got keys: {list(payload.keys())}")
elif isinstance(payload, list):
    models = payload
else:
    raise RuntimeError(f"[FAIL FAST] Unexpected payload type: {type(payload)}. Expected dict or list.")

print(f"[Ingestion] Successfully fetched {len(models)} models for provider '{resolved_provider}'.")

# 4. Storage & Snapshot Writer
if resolved_save:
    # Standard dataset naming: <provider>-models-<MMDDYY>.json (e.g. featherless-models-083126.json)
    date_str = datetime.now().strftime("%m%d%y")
    filename = f"{resolved_provider.lower()}-models-{date_str}.json"
    
    # Ensure datasets/ directory exists
    datasets_dir = Path("datasets")
    datasets_dir.mkdir(parents=True, exist_ok=True)
    output_path = datasets_dir / filename
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"[Snapshot Vault] Saved permanent dataset snapshot to: {output_path}")
else:
    # Ephemeral session output
    output_path = Path("models-list.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"[Ephemeral Session] Opted out of permanent snapshot. Saved session file to: {output_path}")

[Ingestion] Fetching models for 'nvidia' from: https://integrate.api.nvidia.com/v1/models
[Ingestion] Successfully fetched 83 models for provider 'nvidia'.
[Snapshot Vault] Saved permanent dataset snapshot to: datasets/nvidia-models-083126.json


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=cf941940-56b9-494d-b9e4-8b53a91f8a1e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>